# Solar Filament Segmentation Challenge 2026 - YOLO26 Full-Resolution Baseline

This notebook trains and evaluates a YOLO26 segmentation model on the original 2048 x 2048 images while staying within a 30 GiB RAM and 57.6 GiB disk budget.

## Resource settings

- **2048 x 2048 input resolution** for training, validation, and test inference.
- **Both available T4 GPUs** for training, with a global batch of 2 (one image per GPU).
- **At most four CPU data workers** during training (two per GPU); CPU preprocessing pools also use at most four single-threaded workers.
- **Adaptive disk cache**: use it only when the actual working volume can fit it with headroom; otherwise disable caching. RAM caching is never used.
- **One image per inference batch** and a bounded validation queue to keep full-resolution masks from accumulating.
- Submission CSV validation reads rows incrementally rather than loading all RLE strings into RAM.

Validation metrics include PQ, SQ, RQ, Dice, IoU, precision, recall, and instance error counts. Test ground truth is never accessed.

## 1. Install dependencies and import libraries

The training configuration uses disk caching rather than a RAM cache. Package installation disables pip's wheel cache to conserve the session disk.

In [ ]:
%pip -q install ultralytics pycocotools scikit-learn tqdm torchmetrics threadpoolctl psutil

In [ ]:
from pathlib import Path
import os
import csv
import gc
import random
import shutil
import time
import warnings
import multiprocessing
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
try:
    import psutil
except ImportError:
    psutil = None

from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from torchmetrics.segmentation import DiceScore
from threadpoolctl import threadpool_limits
from ultralytics import YOLO

# Hide standard Python warnings
warnings.filterwarnings("ignore")

# Hide underlying C-level / ctypes environment logs
os.environ["PYTHONWARNINGS"] = "ignore"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA_DEVICE_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
MAX_GPUS_TO_USE = 2
GPU_DEVICE_IDS = list(range(min(CUDA_DEVICE_COUNT, MAX_GPUS_TO_USE)))
GPU_COUNT_USED = len(GPU_DEVICE_IDS)

if GPU_COUNT_USED >= 2:
    TRAIN_DEVICE = GPU_DEVICE_IDS
elif GPU_COUNT_USED == 1:
    TRAIN_DEVICE = GPU_DEVICE_IDS[0]
else:
    TRAIN_DEVICE = "cpu"

# Ultralytics prediction uses one explicit device; distributed training below
# uses both GPUs when they are available.
INFERENCE_DEVICE = GPU_DEVICE_IDS[0] if GPU_COUNT_USED else "cpu"

print(f"Available GPU IDs: {GPU_DEVICE_IDS}")
print(f"Inference device: {INFERENCE_DEVICE}")
if CUDA_DEVICE_COUNT:
    for gpu_id in GPU_DEVICE_IDS:
        print(f"GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
    print(f"CUDA devices available: {CUDA_DEVICE_COUNT}; using {GPU_COUNT_USED}")

CPU_COUNT = os.cpu_count() or 1
print(f"Logical CPU count: {CPU_COUNT}")

## 2. Configure paths and experiment

The notebook caps CPU worker processes at four. Multi-GPU training uses two data workers per GPU (four total); custom preprocessing stages use a four-process pool with one thread per worker. Images and inference masks remain at 2048 x 2048, while batches and queues stay small to protect the 30 GiB RAM limit.

In [ ]:
SEED = 42

NUM_EPOCHS = 150

VALIDATION_FRACTION = 0.2
PATIENCE = 10

IMAGE_SIZE = 2048
NUM_WORKERS = 4  # Hard cap for custom CPU worker pools.
WORKER_CPU_THREADS = 1

# Custom callbacks are not forwarded into the current Ultralytics DDP relaunch.
# Keep this experiment single-GPU so the PQ callback executes. Set False to restore multi-GPU DDP.
PQ_CALLBACK_ENABLED = True
PQ_CALLBACK_EVERY = 10
PQ_CALLBACK_IMAGES = 32

if PQ_CALLBACK_ENABLED and GPU_COUNT_USED > 0:
    TRAIN_GPU_COUNT = 1
    TRAIN_DEVICE = GPU_DEVICE_IDS[0]
elif GPU_COUNT_USED > 0:
    TRAIN_GPU_COUNT = GPU_COUNT_USED
    TRAIN_DEVICE = GPU_DEVICE_IDS
else:
    TRAIN_GPU_COUNT = 0
    TRAIN_DEVICE = "cpu"

YOLO_WORKERS_PER_GPU = (
    min(NUM_WORKERS // max(TRAIN_GPU_COUNT, 1), 4)
    if TRAIN_GPU_COUNT > 0
    else 0
)
BATCH_SIZE = TRAIN_GPU_COUNT if TRAIN_GPU_COUNT > 0 else 1  # One image per GPU (or one on CPU).
print(f"Training device(s): {TRAIN_DEVICE}")
print(f"Global training batch: {BATCH_SIZE} ({YOLO_WORKERS_PER_GPU} workers per GPU)")

# One native-resolution image at a time keeps full-size masks bounded.
VAL_BATCH_SIZE = 1
MAX_PENDING_VALIDATION = NUM_WORKERS * 2
MAX_DETECTIONS = 100

RAM_LIMIT_GIB = 30.0
RAM_RESERVE_GIB = 6.0
DISK_LIMIT_GIB = 57.6
DISK_RESERVE_GIB = 10.0  # Preferred reserve; reduced only if the mounted volume requires it.
MIN_DISK_RESERVE_GIB = 4.0
CACHE_SAFETY_FACTOR = 1.25
TRAIN_OUTPUT_RESERVE_GIB = 2.0
GIB = 1024 ** 3
YOLO_CACHE_MODE = "disk"  # Preflight switches this to False if disk cache cannot fit.
ACTIVE_DISK_RESERVE_GIB = DISK_RESERVE_GIB

MIN_COMPONENT_AREA = 20

# Crop-based U-Net mask refinement (trained only from train_records).
USE_REFINER = True
REFINER_CROP_SIZE = 256
REFINER_CONTEXT = 1.8
REFINER_MIN_CROP = 96
REFINER_EPOCHS = 15
REFINER_BATCH_SIZE = 8
REFINER_LR = 3e-4
REFINER_MASK_THRESHOLD = 0.5
REFINER_INFERENCE_BATCH = 8
REFINER_PRETRAINED = True
YOLO_MODEL_NAME = "yolo26s-seg.pt"

YOLO_DATA_DIR = Path.cwd() / "yolo_filament_dataset"
RUNS_DIR = Path.cwd() / "yolo_runs"
RUN_NAME = "yolo26s_filament_full_resolution"
DATA_YAML = YOLO_DATA_DIR / "filament.yaml"

SUBMISSION_PATH = Path.cwd() / "solar_filament_yolo26s_submission.csv"
FINAL_METRICS_PATH = Path.cwd() / "solar_filament_yolo26s_validation_metrics.csv"
PER_IMAGE_METRICS_PATH = Path.cwd() / "solar_filament_yolo26s_per_image_metrics.csv"

REBUILD_YOLO_DATASET = True
CREATE_SUBMISSION = True

CONFIDENCE_CANDIDATES = (0.15, 0.25, 0.35, 0.45, 0.55)
PQ_MATCH_IOU = 0.50


def _read_cgroup_memory_pair(limit_path, current_path):
    try:
        limit_text = Path(limit_path).read_text().strip()
        current_text = Path(current_path).read_text().strip()
        if limit_text == "max":
            return None
        limit_bytes = int(limit_text)
        current_bytes = int(current_text)
        if limit_bytes >= (1 << 60):  # cgroup v1 unlimited sentinel
            return None
        return max(0, limit_bytes - current_bytes)
    except (OSError, ValueError):
        return None


def available_ram_bytes():
    for limit_path, current_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        available = _read_cgroup_memory_pair(limit_path, current_path)
        if available is not None:
            return available
    if psutil is not None:
        try:
            return int(psutil.virtual_memory().available)
        except Exception:
            pass
    return None


def require_ram_headroom(stage, minimum_gib):
    available = available_ram_bytes()
    if available is None:
        print(f"[RAM] Could not read a runtime memory limit before {stage}; using bounded batches/workers.")
        return
    available_gib = available / GIB
    print(f"[RAM] Available before {stage}: {available_gib:.1f} GiB")
    if available_gib < minimum_gib:
        raise MemoryError(
            f"Only {available_gib:.1f} GiB RAM is available before {stage}; "
            f"need at least {minimum_gib:.1f} GiB to preserve headroom under the "
            f"{RAM_LIMIT_GIB:.1f} GiB session limit. Free memory or restart the session."
        )


def directory_size_bytes(root):
    root = Path(root)
    if not root.exists():
        return 0
    total = 0
    for item in root.rglob("*"):
        if item.is_file() and not item.is_symlink():
            try:
                total += item.stat().st_size
            except OSError:
                pass
    return total


def preflight_training_resources(train_split, val_split, image_dir):
    global YOLO_CACHE_MODE, ACTIVE_DISK_RESERVE_GIB

    require_ram_headroom("dataset preparation", minimum_gib=RAM_RESERVE_GIB + 2.0)
    records = pd.concat([train_split, val_split], ignore_index=True)

    # Conservative cache estimate: RGB uint8 array per YOLO record at full size.
    estimated_cache_bytes = int(
        len(records) * IMAGE_SIZE * IMAGE_SIZE * 3 * CACHE_SAFETY_FACTOR
    )
    # Copies are only used if symlink creation is unavailable; count duplicate records too.
    copy_bytes = 0
    for file_name in records["file_name"]:
        source = Path(image_dir) / file_name
        if source.is_file():
            copy_bytes += source.stat().st_size

    output_reserve = int(TRAIN_OUTPUT_RESERVE_GIB * GIB)
    workspace_bytes = directory_size_bytes(Path.cwd())
    disk = shutil.disk_usage(YOLO_DATA_DIR.parent)
    effective_budget_bytes = min(int(DISK_LIMIT_GIB * GIB), int(disk.total))

    # Prefer disk caching with the full reserve. If the mounted filesystem is
    # smaller than the advertised session quota, turn caching off (never RAM-on)
    # and only reduce the reserve as far as 4 GiB.
    cache_options = (("disk", estimated_cache_bytes), (False, 0))
    reserve_options = [
        value for value in (DISK_RESERVE_GIB, 8.0, 6.0, MIN_DISK_RESERVE_GIB)
        if value <= DISK_RESERVE_GIB
    ]
    selected_plan = None

    for cache_mode, cache_bytes in cache_options:
        for reserve_gib in reserve_options:
            reserve_bytes = int(reserve_gib * GIB)
            planned_new_bytes = cache_bytes + copy_bytes + output_reserve + reserve_bytes
            if (
                planned_new_bytes <= effective_budget_bytes
                and workspace_bytes + planned_new_bytes <= effective_budget_bytes
                and disk.free >= planned_new_bytes
            ):
                selected_plan = (cache_mode, reserve_gib, planned_new_bytes, cache_bytes)
                break
        if selected_plan is not None:
            break

    if selected_plan is None:
        no_cache_minimum = (
            copy_bytes
            + output_reserve
            + int(MIN_DISK_RESERVE_GIB * GIB)
        )
        raise OSError(
            "Insufficient writable disk even with YOLO image caching disabled. "
            f"Filesystem capacity: {disk.total / GIB:.1f} GiB; free: {disk.free / GIB:.1f} GiB; "
            f"workspace: {workspace_bytes / GIB:.1f} GiB; minimum planned use: "
            f"{no_cache_minimum / GIB:.1f} GiB (includes a {MIN_DISK_RESERVE_GIB:.1f} GiB reserve). "
            "Free disk space or select a larger working volume."
        )

    YOLO_CACHE_MODE, ACTIVE_DISK_RESERVE_GIB, planned_new_bytes, cache_bytes = selected_plan
    print(
        f"[DISK] Cache mode: {YOLO_CACHE_MODE}; estimated cache: {cache_bytes / GIB:.1f} GiB; "
        f"worst-case image copies: {copy_bytes / GIB:.2f} GiB."
    )
    print(
        f"[DISK] Planned additional use: {planned_new_bytes / GIB:.1f} GiB; "
        f"reserved free space: {ACTIVE_DISK_RESERVE_GIB:.1f} GiB; "
        f"filesystem free: {disk.free / GIB:.1f} GiB."
    )
    if YOLO_CACHE_MODE is False:
        print("[DISK] Full-resolution training will continue without image caching; RAM caching is disabled.")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_main_process_threads():
    cv2.setNumThreads(WORKER_CPU_THREADS)
    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass


configure_main_process_threads()
set_seed()

# --- Logging Framework Configurations ---
print(f"Random seed initialization: {SEED}")
print(f"Split ratio: Train = {1.0 - VALIDATION_FRACTION:.2f} / Validation = {VALIDATION_FRACTION:.2f}")
print(f"Training epochs: {NUM_EPOCHS} (Early stopping patience: {PATIENCE})")
print(f"Input size: {IMAGE_SIZE} x {IMAGE_SIZE}")
print(f"Custom CPU worker cap: {NUM_WORKERS}")
print(f"YOLO data workers per GPU: {YOLO_WORKERS_PER_GPU}")
print(f"YOLO global batch size: {BATCH_SIZE}")
print(f"Inference batch size: {VAL_BATCH_SIZE}")
print("Image cache mode: selected by disk preflight; RAM caching is disabled.")

In [ ]:
def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]

    for parent in candidates:
        if not parent.exists():
            continue

        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train" / "train_images").is_dir() and (root / "test" / "test_images").is_dir():
                return root

    raise FileNotFoundError("Could not find MAGFiLO_1.0_Kaggle_2026 with train and test image folders.")

DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")
print(f"YOLO model: {YOLO_MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Global training batch: {BATCH_SIZE} ({YOLO_WORKERS_PER_GPU} workers per GPU)")
print(f"Validation fraction: {VALIDATION_FRACTION:.0%}")
print(f"PQ matching IoU threshold: {PQ_MATCH_IOU:.2f}")

## 3. Load annotations and create a leakage-safe split

The split is performed using the physical filename rather than the COCO record ID. Multiple annotation records for the same JPEG remain on one side of the split.

In [ ]:
coco = COCO(str(ANNOTATION_PATH))
image_ids = list(coco.imgs.keys())
records = pd.DataFrame([{"image_id": image_id, "file_name": coco.imgs[image_id]["file_name"]} for image_id in image_ids])

splitter = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(records, groups=records["file_name"]))

train_records = records.iloc[train_idx].reset_index(drop=True)
val_records = records.iloc[val_idx].reset_index(drop=True)

train_files = set(train_records["file_name"])
val_files = set(val_records["file_name"])

assert train_files.isdisjoint(val_files), "A physical image appears in both splits."

print(f"Training records: {len(train_records):,} across {len(train_files):,} physical images")
print(f"Validation records: {len(val_records):,} across {len(val_files):,} physical images")
print(f"Train/validation filename overlap: {len(train_files & val_files)}")

## 4. Build the YOLO segmentation dataset

COCO masks are converted to YOLO polygons. Input JPEGs are linked into the working dataset when possible, avoiding a second copy; a conservative copy-size estimate is included in the disk preflight.

In [ ]:
def configure_worker_process():
    cv2.setNumThreads(WORKER_CPU_THREADS)
    try:
        torch.set_num_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass
    try:
        torch.set_num_interop_threads(WORKER_CPU_THREADS)
    except RuntimeError:
        pass
    try:
        threadpool_limits(limits=WORKER_CPU_THREADS)
    except Exception:
        pass

    if hasattr(os, "sched_getaffinity"):
        available_cpus = sorted(os.sched_getaffinity(0))
        if available_cpus:
            identity = multiprocessing.current_process()._identity
            worker_index = identity[0] - 1 if identity else 0
            cpu = available_cpus[worker_index % len(available_cpus)]
            if hasattr(os, "sched_setaffinity"):
                os.sched_setaffinity(0, {cpu})

def mask_to_yolo_segments(mask, min_area):
    mask = (mask > 0).astype(np.uint8)
    height, width = mask.shape
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    segments = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue

        component = (labels == label).astype(np.uint8)
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)
        if len(contour) < 3 or cv2.contourArea(contour) <= 0:
            continue
            
        # Ensure the component actually has a valid structural bounding box ---
        bx, by, bw, bh = cv2.boundingRect(contour)
        if bw < 2 or bh < 2:  # Rejects single-pixel lines or noise
            continue

        points = contour.reshape(-1, 2).astype(np.float32)
        points[:, 0] /= width
        points[:, 1] /= height
        points = np.clip(points, 0.0, 1.0)
        
        # Re-verify normalized bounding box dimensions to protect Pillow ---
        if len(points) >= 3:
            x_min, y_min = points[:, 0].min(), points[:, 1].min()
            x_max, y_max = points[:, 0].max(), points[:, 1].max()
            
            # If the normalization completely flattened a microscopic edge, ignore it
            if (x_max - x_min) <= 1e-4 or (y_max - y_min) <= 1e-4:
                continue
                
            segments.append(points)

    return segments

def worker_process_record(row_dict, annotation_path_str, min_area, train_image_dir_str, image_dir_str, label_dir_str):
    global _local_coco
    configure_worker_process()

    if "_local_coco" not in globals():
        import sys
        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_coco = COCO(annotation_path_str)
        sys.stdout = old_stdout

    train_img_dir = Path(train_image_dir_str)
    img_out_dir = Path(image_dir_str)
    lbl_out_dir = Path(label_dir_str)

    image_id = row_dict["image_id"]
    file_name = row_dict["file_name"]
    source_path = train_img_dir / file_name

    if not source_path.is_file():
        return 0

    source_suffix = source_path.suffix
    source_stem = Path(file_name).stem
    sample_stem = f"{image_id}_{source_stem}"

    image_path = img_out_dir / f"{sample_stem}{source_suffix}"
    label_path = lbl_out_dir / f"{sample_stem}.txt"

    image_path.parent.mkdir(parents=True, exist_ok=True)
    if image_path.exists() or image_path.is_symlink():
        image_path.unlink()

    try:
        image_path.symlink_to(source_path.resolve())
    except OSError:
        shutil.copy2(source_path, image_path)

    annotations = _local_coco.loadAnns(_local_coco.getAnnIds(imgIds=[image_id]))
    lines = []

    for annotation in annotations:
        mask = _local_coco.annToMask(annotation).astype(np.uint8)
        for points in mask_to_yolo_segments(mask, min_area):
            flat = points.reshape(-1)
            values = ["0"]
            values.extend(f"{value:.6f}" for value in flat)
            lines.append(" ".join(values))

    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    return len(lines)

def write_yolo_split_parallel(split_records, split_name):
    print(f"\n[START] Building YOLO split: '{split_name}'", flush=True)
    image_dir = YOLO_DATA_DIR / "images" / split_name
    label_dir = YOLO_DATA_DIR / "labels" / split_name

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    records_list = split_records[["image_id", "file_name"]].to_dict(orient="records")
    worker_task = partial(
        worker_process_record,
        annotation_path_str=str(ANNOTATION_PATH),
        min_area=MIN_COMPONENT_AREA,
        train_image_dir_str=str(TRAIN_IMAGE_DIR),
        image_dir_str=str(image_dir),
        label_dir_str=str(label_dir),
    )

    empty_labels = 0
    total_instances = 0

    with multiprocessing.Pool(processes=NUM_WORKERS, initializer=configure_worker_process) as pool:
        results = tqdm(
            pool.imap_unordered(worker_task, records_list, chunksize=4),
            total=len(records_list),
            desc=f"Preparing YOLO {split_name}",
        )

        for idx, instance_count in enumerate(results, start=1):
            total_instances += instance_count
            if instance_count == 0:
                empty_labels += 1

            if idx == 1 or idx % 100 == 0 or idx == len(records_list):
                print(f"[{split_name.upper()} Pre-Processing] {idx:,}/{len(records_list):,} | polygons: {total_instances:,} | empty: {empty_labels:,}", flush=True)

    print(f"[FINISHED] {split_name}: {len(records_list):,} records | {total_instances:,} polygons | empty labels: {empty_labels:,}\n", flush=True)

if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    print(f"[CLEANUP] Removing old directory: {YOLO_DATA_DIR}", flush=True)
    shutil.rmtree(YOLO_DATA_DIR)

preflight_training_resources(train_records, val_records, TRAIN_IMAGE_DIR)
write_yolo_split_parallel(train_records, "train")
write_yolo_split_parallel(val_records, "val")

DATA_YAML.write_text(
    f"path: '{YOLO_DATA_DIR.as_posix()}'\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: filament\n",
    encoding="utf-8",
)

print(f"YOLO dataset configuration: {DATA_YAML}")
print(DATA_YAML.read_text())

## 5. Train YOLO26s-seg at full resolution

With the PQ callback enabled, training uses one GPU so the callback is preserved. Every 10 epochs it evaluates a fixed 32-image validation subset at the configured confidence candidates, records PQ plus one-to-many/many-to-one counts, and saves the best-PQ YOLO checkpoint. Set `PQ_CALLBACK_ENABLED = False` to restore multi-GPU DDP training; that mode does not run this notebook callback. Disk caching is enabled only when the actual working volume can fit the estimated cache and reserve. If it cannot, caching is disabled (never switched to RAM).

In [ ]:
# The PQ callback evaluates a deterministic validation subset periodically.
pq_callback_state = {
    "best_pq": -float("inf"),
    "best_checkpoint": None,
    "last_eval_epoch": None,
    "history": [],
}
pq_callback_records = (
    val_records.sample(
        n=min(PQ_CALLBACK_IMAGES, len(val_records)), random_state=SEED
    )
    .sort_values("file_name")
    .to_dict(orient="records")
)

def _pq_callback_components(binary_mask):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8), connectivity=8
    )
    return [
        labels == label
        for label in range(1, count)
        if int(stats[label, cv2.CC_STAT_AREA]) >= MIN_COMPONENT_AREA
    ]

def _pq_callback_score(target_instances, prediction_instances):
    target_count = len(target_instances)
    prediction_count = len(prediction_instances)
    if target_count == 0 and prediction_count == 0:
        return 1.0, 0, 0

    iou_matrix = np.zeros((target_count, prediction_count), dtype=np.float32)
    for target_index, target_mask in enumerate(target_instances):
        target_area = int(target_mask.sum())
        if target_area == 0:
            continue
        for prediction_index, prediction_mask in enumerate(prediction_instances):
            intersection = int(np.logical_and(target_mask, prediction_mask).sum())
            if intersection:
                union = target_area + int(prediction_mask.sum()) - intersection
                iou_matrix[target_index, prediction_index] = intersection / union

    relations = iou_matrix > PQ_MATCH_IOU
    one_to_many = int((relations.sum(axis=1) > 1).sum())
    many_to_one = int((relations.sum(axis=0) > 1).sum())
    pairs = [
        (float(iou_matrix[target_index, prediction_index]), target_index, prediction_index)
        for target_index in range(target_count)
        for prediction_index in range(prediction_count)
        if iou_matrix[target_index, prediction_index] > PQ_MATCH_IOU
    ]
    pairs.sort(reverse=True)
    matched_targets, matched_predictions, matched_iou_sum = set(), set(), 0.0
    for iou, target_index, prediction_index in pairs:
        if target_index in matched_targets or prediction_index in matched_predictions:
            continue
        matched_targets.add(target_index)
        matched_predictions.add(prediction_index)
        matched_iou_sum += iou

    true_positives = len(matched_targets)
    false_positives = prediction_count - true_positives
    false_negatives = target_count - true_positives
    denominator = true_positives + 0.5 * false_positives + 0.5 * false_negatives
    pq = matched_iou_sum / denominator if denominator else 1.0
    return float(pq), one_to_many, many_to_one

def pq_fit_epoch_callback(trainer):
    if not PQ_CALLBACK_ENABLED:
        return
    # This run is single-GPU when enabled; retain a guard for accidental DDP execution.
    if getattr(trainer, "rank", -1) not in (-1, 0):
        return

    epoch = int(getattr(trainer, "epoch", -1)) + 1
    stopped = bool(getattr(trainer, "stop", False))
    final_epoch = epoch >= int(getattr(trainer, "epochs", epoch))
    due = (epoch % PQ_CALLBACK_EVERY == 0) or final_epoch or stopped
    if not due or pq_callback_state["last_eval_epoch"] == epoch:
        return

    checkpoint = Path(trainer.last)
    if not checkpoint.is_file():
        print(f"[PQ callback] Checkpoint missing at epoch {epoch}; skipping.", flush=True)
        return

    eval_model = YOLO(str(checkpoint))
    per_threshold = {threshold: [] for threshold in CONFIDENCE_CANDIDATES}
    relation_totals = {threshold: {"one_to_many": 0, "many_to_one": 0} for threshold in CONFIDENCE_CANDIDATES}
    evaluation_start = time.perf_counter()
    valid_images = 0

    try:
        for row in pq_callback_records:
            image_path = TRAIN_IMAGE_DIR / row["file_name"]
            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue

            source = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
            result = eval_model.predict(
                source=[source], imgsz=IMAGE_SIZE, conf=min(CONFIDENCE_CANDIDATES),
                iou=0.7, retina_masks=True, device=INFERENCE_DEVICE, batch=1,
                max_det=MAX_DETECTIONS, verbose=False,
            )[0]

            predicted_masks = []
            if result.masks is not None and result.boxes is not None:
                masks = result.masks.data.detach().float().cpu().numpy()
                confidences = result.boxes.conf.detach().float().cpu().numpy()
                for mask, confidence in zip(masks, confidences):
                    if mask.shape != image.shape:
                        mask = cv2.resize(
                            mask, (image.shape[1], image.shape[0]),
                            interpolation=cv2.INTER_NEAREST,
                        )
                    predicted_masks.append((mask >= 0.5, float(confidence)))

            target_instances = []
            for annotation in coco.imgToAnns.get(row["image_id"], []):
                target_mask = coco.annToMask(annotation).astype(bool)
                if target_mask.any():
                    target_instances.append(target_mask)
            valid_images += 1

            for threshold in CONFIDENCE_CANDIDATES:
                union_mask = np.zeros(image.shape, dtype=bool)
                for mask, confidence in predicted_masks:
                    if confidence >= threshold:
                        union_mask |= mask
                prediction_instances = _pq_callback_components(union_mask)
                pq, one_to_many, many_to_one = _pq_callback_score(
                    target_instances, prediction_instances
                )
                per_threshold[threshold].append(pq)
                relation_totals[threshold]["one_to_many"] += one_to_many
                relation_totals[threshold]["many_to_one"] += many_to_one

        if valid_images == 0:
            print(f"[PQ callback] No readable images at epoch {epoch}; skipping.", flush=True)
            return

        threshold_scores = {
            threshold: float(np.mean(scores))
            for threshold, scores in per_threshold.items() if scores
        }
        selected_confidence = max(threshold_scores, key=threshold_scores.get)
        current_pq = threshold_scores[selected_confidence]
        row = {
            "epoch": epoch, "PQ": current_pq, "confidence": selected_confidence,
            "one_to_many": relation_totals[selected_confidence]["one_to_many"],
            "many_to_one": relation_totals[selected_confidence]["many_to_one"],
            "images": valid_images,
            "evaluation_seconds": time.perf_counter() - evaluation_start,
        }
        pq_callback_state["history"].append(row)
        pq_callback_state["last_eval_epoch"] = epoch

        if current_pq > pq_callback_state["best_pq"]:
            best_path = Path(trainer.save_dir) / "weights" / "best_pq.pt"
            shutil.copy2(checkpoint, best_path)
            pq_callback_state["best_pq"] = current_pq
            pq_callback_state["best_checkpoint"] = str(best_path)
            row["best"] = True
        else:
            row["best"] = False

        print(
            f"[PQ callback] epoch {epoch}: PQ={current_pq:.4f} "
            f"(conf={selected_confidence:.2f}, "
            f"1-to-many={row['one_to_many']}, "
            f"many-to-1={row['many_to_one']}, "
            f"{row['evaluation_seconds']:.1f}s)"
            + (" [best]" if row["best"] else ""),
            flush=True,
        )
    finally:
        del eval_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Building pretrained YOLO26s segmentation model...")
model = YOLO(YOLO_MODEL_NAME)
if PQ_CALLBACK_ENABLED:
    model.add_callback("on_fit_epoch_end", pq_fit_epoch_callback)

require_ram_headroom("training", minimum_gib=RAM_RESERVE_GIB + 6.0)
training_start = time.perf_counter()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=TRAIN_DEVICE,
    workers=YOLO_WORKERS_PER_GPU,
    cache=YOLO_CACHE_MODE,
    max_det=MAX_DETECTIONS,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    amp=(DEVICE.type == "cuda"),
    seed=SEED,
    deterministic=True,
    fliplr=0.5,
    flipud=0.5,
    degrees=180.0,
    translate=0.03,
    scale=0.05,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.3,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    plots=True,
    verbose=True,
)

training_time = time.perf_counter() - training_start

try:
    run_dir = Path(model.trainer.save_dir)
except AttributeError:
    run_dir = RUNS_DIR / RUN_NAME

best_checkpoint = run_dir / "weights" / "best.pt"
pq_checkpoint = Path(pq_callback_state["best_checkpoint"]) if pq_callback_state["best_checkpoint"] else None
if pq_checkpoint is not None and pq_checkpoint.is_file():
    best_checkpoint = pq_checkpoint
    print(f"Selecting best-PQ checkpoint: {best_checkpoint} (PQ={pq_callback_state['best_pq']:.4f})")
else:
    print("No PQ callback checkpoint was produced; falling back to Ultralytics best.pt.")
if not best_checkpoint.is_file():
    raise FileNotFoundError(f"Best checkpoint was not created: {best_checkpoint}")

pq_callback_history = pd.DataFrame(pq_callback_state["history"])
if not pq_callback_history.empty:
    pq_callback_history.to_csv(run_dir / "pq_callback_metrics.csv", index=False)

del train_results
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = YOLO(str(best_checkpoint))

print(f"\n[SUCCESS] Training complete!\nRun directory: {run_dir}\nBest checkpoint: {best_checkpoint}\nTraining time: {training_time / 60:.1f} min", flush=True)

## Show Training History

In [ ]:
yolo_visual_groups = {
    "Overall Performance & Loss Tracking": [
        "results.png"
    ],
    "Bounding Box Localization Metrics": [
        "BoxF1_curve.png",
        "BoxPR_curve.png",
        "BoxP_curve.png",
        "BoxR_curve.png"
    ],
    "Instance Segmentation Mask Metrics": [
        "MaskF1_curve.png",
        "MaskPR_curve.png",
        "MaskP_curve.png",
        "MaskR_curve.png"
    ],
    "Confusion Matrices": [
        "confusion_matrix.png",
        "confusion_matrix_normalized.png"
    ],
    "Labels": [
        "labels.jpg"
    ],
    "Training Dataset Batches & Anchors": [
        "train_batch0.jpg",
        "train_batch1.jpg",
        "train_batch2.jpg"
    ],
    "Validation Predictions vs Ground Truth": [
        "val_batch0_labels.jpg",
        "val_batch0_pred.jpg",
        "val_batch1_labels.jpg",
        "val_batch1_pred.jpg",
        "val_batch2_labels.jpg",
        "val_batch2_pred.jpg"
    ]
}


display_settings = {
    "Overall Performance & Loss Tracking": {
        "cols": 1,
        "col_width": 16,
        "row_height": 9,
    },
    "Bounding Box Localization Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Instance Segmentation Mask Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Confusion Matrices": {
        "cols": 1,
        "col_width": 10,
        "row_height": 8,
    },
    "Labels": {
        "cols": 1,
        "col_width": 12,
        "row_height": 8,
    },
    "Training Dataset Batches & Anchors": {
        "cols": 3,
        "col_width": 6,
        "row_height": 5,
    },
    "Validation Predictions vs Ground Truth": {
        "cols": 2,
        "col_width": 6,
        "row_height": 5,
    },
}


print(
    f"Loading generated artifact visuals from: {run_dir}\n"
    + "=" * 70
)


for group_name, plot_files in yolo_visual_groups.items():
    valid_files = [
        f
        for f in plot_files
        if (run_dir / f).is_file()
    ]

    if not valid_files:
        continue

    print(f"\n▶ Displaying Group: {group_name}")
    print("-" * 70)

    settings = display_settings[group_name]

    cols = settings["cols"]
    col_width = settings["col_width"]
    row_height = settings["row_height"]

    num_plots = len(valid_files)
    num_rows = (num_plots + cols - 1) // cols

    fig, axes = plt.subplots(
        num_rows,
        cols,
        figsize=(
            col_width * cols,
            row_height * num_rows,
        ),
    )

    axes = np.atleast_1d(axes).flatten()

    for idx, plot_name in enumerate(valid_files):
        img = cv2.imread(str(run_dir / plot_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[idx].imshow(img)
        axes[idx].set_title(
            plot_name,
            fontsize=11,
            fontweight="bold",
            pad=8,
        )
        axes[idx].axis("off")

    for idx in range(num_plots, len(axes)):
        axes[idx].axis("off")

    fig.tight_layout()
    plt.show()

In [ ]:
history_path = (
    run_dir / "results.csv"
)

if not history_path.is_file():
    raise FileNotFoundError(
        f"YOLO training history not found: "
        f"{history_path}"
    )

history = pd.read_csv(
    history_path
)

history.columns = [
    column.strip()
    for column in history.columns
]

print(
    "Available history columns:"
)

print(
    ", ".join(history.columns)
)

epochs = np.arange(
    1,
    len(history) + 1,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
)

for column, label in [
    (
        "train/seg_loss",
        "Train segmentation loss",
    ),
    (
        "val/seg_loss",
        "Validation segmentation loss",
    ),
]:
    if column in history.columns:
        axes[0].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[0].set(
    title="YOLO segmentation loss",
    xlabel="Epoch",
    ylabel="Loss",
)

axes[0].legend()
axes[0].grid(alpha=0.25)

for column, label in [
    (
        "metrics/mAP50(M)",
        "Mask mAP50",
    ),
    (
        "metrics/mAP50-95(M)",
        "Mask mAP50-95",
    ),
]:
    if column in history.columns:
        axes[1].plot(
            epochs,
            history[column],
            marker="o",
            label=label,
        )

axes[1].set(
    title="YOLO validation mask metrics",
    xlabel="Epoch",
    ylabel="Metric",
    ylim=(0, 1),
)

axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

print(
    f"Best checkpoint: "
    f"{best_checkpoint}"
)

## 7. Prediction and metric helpers

The prediction path is shared by validation and submission:

1. grayscale image → 3 identical pretrained-model channels
2. YOLO instance segmentation
3. confidence filtering
4. union of selected masks
5. connected-component splitting
6. evaluation or RLE encoding

In [ ]:
def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(f"Expected grayscale image, got shape {image.shape}")
    return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

def get_test_image_paths():
    paths = [path for path in TEST_IMAGE_DIR.iterdir() if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"}]
    return sorted(paths, key=lambda path: path.name)

def predict_instances_batch(images, min_confidence=0.15, apply_refiner=None):
    sources = [to_yolo_image(image) for image in images]
    results = model.predict(
        source=sources, imgsz=IMAGE_SIZE, conf=min_confidence, iou=0.7, retina_masks=True,
        device=INFERENCE_DEVICE, batch=len(sources), max_det=MAX_DETECTIONS, verbose=False
    )
    outputs = []

    for image, result in zip(images, results):
        if result.masks is None or result.boxes is None:
            outputs.append([])
            continue

        masks = result.masks.data.detach().float().cpu().numpy()
        confidences = result.boxes.conf.detach().float().cpu().numpy()
        native_height, native_width = image.shape
        instances = []

        for mask, confidence in zip(masks, confidences):
            if mask.shape != image.shape:
                mask = cv2.resize(mask, (native_width, native_height), interpolation=cv2.INTER_NEAREST)
            instances.append({"mask": mask >= 0.5, "confidence": float(confidence)})
        outputs.append(instances)

    if apply_refiner is None:
        apply_refiner = USE_REFINER
    if apply_refiner:
        outputs = [
            refine_instances_for_image(image, instances)
            for image, instances in zip(images, outputs)
        ]

    return outputs

def predict_instances(image, min_confidence=0.15, apply_refiner=None):
    return predict_instances_batch(
        [image], min_confidence=min_confidence, apply_refiner=apply_refiner
    )[0]

def union_instances(instances, confidence_threshold):
    selected = [instance for instance in instances if instance["confidence"] >= confidence_threshold]
    if not selected:
        return None

    foreground = np.zeros_like(selected[0]["mask"], dtype=bool)
    for instance in selected:
        foreground |= instance["mask"]
    return foreground

def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask.astype(np.uint8), connectivity=8)
    components = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        components.append(labels == label)

    def component_key(component):
        ys, xs = np.where(component)
        return (-int(component.sum()), int(ys.min()) if len(ys) else 0, int(xs.min()) if len(xs) else 0)

    return sorted(components, key=component_key)

def build_final_prediction_instances(instances, confidence_threshold):
    combined = union_instances(instances, confidence_threshold)
    if combined is None:
        return []
    return split_into_filaments(combined, MIN_COMPONENT_AREA)

def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")


### 7.1 Train and apply the crop refiner

The refiner learns from ground-truth crops from the training split only. At inference, each YOLO mask supplies a crop window; the refiner predicts a cleaner mask inside that crop. Because both validation and submission use `predict_instances_batch`, they receive the same refinement. Set `USE_REFINER = False` in the configuration cell to compare with the original YOLO masks. Run this cell before validation or submission.


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights


def square_bounds(mask, context=REFINER_CONTEXT, minimum=REFINER_MIN_CROP):
    height, width = mask.shape
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        side = min(minimum, height, width)
        return 0, 0, side, side

    center_x = (int(xs.min()) + int(xs.max()) + 1) / 2
    center_y = (int(ys.min()) + int(ys.max()) + 1) / 2
    side = int(np.ceil(max(int(xs.max()) - int(xs.min()) + 1,
                           int(ys.max()) - int(ys.min()) + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0 = int(round(center_x - side / 2))
    y0 = int(round(center_y - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0:
        x1 -= x0
        x0 = 0
    if y0 < 0:
        y1 -= y0
        y0 = 0
    if x1 > width:
        x0 -= x1 - width
        x1 = width
    if y1 > height:
        y0 -= y1 - height
        y1 = height
    return int(x0), int(y0), int(x1), int(y1)


print("Building training-only ground-truth crop cache...")
require_ram_headroom("refiner crop cache", minimum_gib=RAM_RESERVE_GIB + 2.0)
refiner_crop_images, refiner_crop_masks = [], []
for row in tqdm(train_records.itertuples(index=False), total=len(train_records)):
    image_path = TRAIN_IMAGE_DIR / row.file_name
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        continue

    for annotation in coco.imgToAnns.get(row.image_id, []):
        mask = coco.annToMask(annotation).astype(np.uint8)
        if int(mask.sum()) < 10:
            continue
        x0, y0, x1, y1 = square_bounds(mask)
        image_crop = cv2.resize(
            image[y0:y1, x0:x1],
            (REFINER_CROP_SIZE, REFINER_CROP_SIZE),
            interpolation=cv2.INTER_LINEAR,
        )
        mask_crop = cv2.resize(
            mask[y0:y1, x0:x1],
            (REFINER_CROP_SIZE, REFINER_CROP_SIZE),
            interpolation=cv2.INTER_NEAREST,
        )
        refiner_crop_images.append(image_crop)
        refiner_crop_masks.append(mask_crop)

if not refiner_crop_images:
    raise RuntimeError("No usable ground-truth crops were created for the refiner.")

crop_images = np.stack(refiner_crop_images).astype(np.uint8, copy=False)
crop_masks = np.stack(refiner_crop_masks).astype(np.uint8, copy=False)
del refiner_crop_images, refiner_crop_masks
gc.collect()
print(
    f"Refiner crops: {len(crop_images):,} | "
    f"array memory: {(crop_images.nbytes + crop_masks.nbytes) / GIB:.2f} GiB"
)


class CropDataset(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images = images
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index].copy()).float() / 255.0
        mask = torch.from_numpy(self.masks[index].copy()).float().unsqueeze(0)

        if self.augment:
            turns = random.randrange(4)
            image = torch.rot90(image, turns, dims=(-2, -1))
            mask = torch.rot90(mask, turns, dims=(-2, -1))
            if random.random() < 0.5:
                image = torch.flip(image, dims=(-1,))
                mask = torch.flip(mask, dims=(-1,))
            if random.random() < 0.5:
                image = image ** random.uniform(0.85, 1.20)
            if random.random() < 0.5:
                image = torch.clamp(image * random.uniform(0.90, 1.10), 0, 1)

        return image.repeat(3, 1, 1), mask


class ConvBlock(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class CropUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            weights = ResNet18_Weights.DEFAULT if pretrained else None
            encoder = resnet18(weights=weights)
        except Exception as error:
            print(f"Could not load pretrained ResNet18 weights ({error}); using random initialization.")
            encoder = resnet18(weights=None)

        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layer1, self.layer2 = encoder.layer1, encoder.layer2
        self.layer3, self.layer4 = encoder.layer3, encoder.layer4
        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = ConvBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = ConvBlock(96, 32)
        self.up0 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.head = nn.Conv2d(16, 1, 1)
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, inputs):
        inputs = (inputs - self.mean) / self.std
        stem = self.stem(inputs)
        layer1 = self.layer1(self.pool(stem))
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        layer4 = self.layer4(layer3)
        decoded = self.dec4(torch.cat([self.up4(layer4), layer3], dim=1))
        decoded = self.dec3(torch.cat([self.up3(decoded), layer2], dim=1))
        decoded = self.dec2(torch.cat([self.up2(decoded), layer1], dim=1))
        decoded = self.dec1(torch.cat([self.up1(decoded), stem], dim=1))
        return self.head(self.up0(decoded))


def soft_dice_loss(logits, targets):
    probabilities = logits.sigmoid()
    intersection = (probabilities * targets).sum((1, 2, 3))
    denominator = probabilities.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()


refiner_dataset = CropDataset(crop_images, crop_masks, augment=True)
refiner_workers = min(NUM_WORKERS, 2)
refiner_loader = DataLoader(
    refiner_dataset,
    batch_size=REFINER_BATCH_SIZE,
    shuffle=True,
    num_workers=refiner_workers,
    pin_memory=DEVICE.type == "cuda",
    worker_init_fn=configure_worker_process if refiner_workers else None,
    persistent_workers=refiner_workers > 0,
)

refiner = CropUNet(pretrained=REFINER_PRETRAINED).to(DEVICE)
refiner_optimizer = torch.optim.AdamW(
    refiner.parameters(), lr=REFINER_LR, weight_decay=1e-4
)
refiner_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    refiner_optimizer, T_max=REFINER_EPOCHS
)
refiner_scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

print(
    f"Training CropUNet for {REFINER_EPOCHS} epochs on "
    f"{len(refiner_dataset):,} training crops, batch={REFINER_BATCH_SIZE}"
)
refiner_start = time.perf_counter()
for epoch in range(1, REFINER_EPOCHS + 1):
    refiner.train()
    epoch_loss_sum = 0.0
    seen_crops = 0

    for images, masks in tqdm(
        refiner_loader, desc=f"Refiner epoch {epoch:02d}/{REFINER_EPOCHS}", leave=False
    ):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        refiner_optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16,
            enabled=DEVICE.type == "cuda"
        ):
            logits = refiner(images)
            bce = nn.functional.binary_cross_entropy_with_logits(logits, masks)
            loss = 0.5 * bce + 0.5 * soft_dice_loss(logits, masks)

        refiner_scaler.scale(loss).backward()
        refiner_scaler.step(refiner_optimizer)
        refiner_scaler.update()
        epoch_loss_sum += float(loss.detach()) * len(images)
        seen_crops += len(images)

    refiner_scheduler.step()
    print(
        f"Refiner epoch {epoch:02d}/{REFINER_EPOCHS} | "
        f"Loss {epoch_loss_sum / max(seen_crops, 1):.4f} | "
        f"LR {refiner_optimizer.param_groups[0]['lr']:.6g}"
    )

refiner.eval()
REFINER_CHECKPOINT = run_dir / "weights" / "refiner.pth"
REFINER_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
torch.save(refiner.state_dict(), REFINER_CHECKPOINT)
refiner_training_seconds = time.perf_counter() - refiner_start
print(
    f"Saved refiner: {REFINER_CHECKPOINT} "
    f"({REFINER_CHECKPOINT.stat().st_size / 1e6:.1f} MB) | "
    f"training time {refiner_training_seconds / 60:.1f} min"
)


def refine_instances_for_image(image, instances):
    if not instances:
        return []

    height, width = image.shape
    refined = [dict(instance) for instance in instances]
    crop_data = []
    crop_targets = []

    for index, instance in enumerate(instances):
        binary_mask = (np.asarray(instance["mask"]) > 0).astype(np.uint8)
        if binary_mask.shape != image.shape:
            binary_mask = cv2.resize(
                binary_mask, (width, height), interpolation=cv2.INTER_NEAREST
            )
        if int(binary_mask.sum()) < 10:
            continue

        x0, y0, x1, y1 = square_bounds(binary_mask)
        crop = cv2.resize(
            image[y0:y1, x0:x1],
            (REFINER_CROP_SIZE, REFINER_CROP_SIZE),
            interpolation=cv2.INTER_LINEAR,
        )
        crop_data.append(crop)
        crop_targets.append((index, x0, y0, x1, y1))

    if not crop_data:
        return refined

    refiner.eval()
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(crop_data), REFINER_INFERENCE_BATCH):
            batch = np.stack(crop_data[start:start + REFINER_INFERENCE_BATCH])
            batch_tensor = torch.from_numpy(batch).float().div_(255.0)
            batch_tensor = batch_tensor.unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)
            logits = refiner(batch_tensor)
            probabilities.extend(logits.sigmoid().squeeze(1).cpu().numpy())

    for probability, (index, x0, y0, x1, y1) in zip(probabilities, crop_targets):
        crop_mask = (probability >= REFINER_MASK_THRESHOLD).astype(np.uint8)
        crop_mask = cv2.resize(
            crop_mask, (x1 - x0, y1 - y0), interpolation=cv2.INTER_NEAREST
        ).astype(bool)
        full_mask = np.zeros((height, width), dtype=bool)
        full_mask[y0:y1, x0:x1] = crop_mask
        refined[index]["mask"] = full_mask

    return refined


## 8. Full-resolution validation

Validation inference processes one 2048 x 2048 image at a time. Its asynchronous CPU queue is capped at eight tasks, and each metric worker uses one thread. This limits simultaneous full-resolution arrays while still overlapping GPU inference with CPU metrics.

In [ ]:
dice_metric = DiceScore(num_classes=2, include_background=False, average="micro", input_format="index")

def calculate_pixel_metrics(prediction, target):
    prediction_tensor = torch.from_numpy(prediction.astype(np.int64)).unsqueeze(0)
    target_tensor = torch.from_numpy(target.astype(np.int64)).unsqueeze(0)

    prediction_has_foreground = bool(prediction.any())
    target_has_foreground = bool(target.any())

    if not prediction_has_foreground and not target_has_foreground:
        dice = 1.0
    else:
        dice = float(dice_metric(prediction_tensor, target_tensor).item())
        if np.isnan(dice):
            dice = 0.0

    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    true_positive = intersection

    false_positive = int(np.logical_and(prediction, ~target).sum())
    false_negative = int(np.logical_and(~prediction, target).sum())

    iou = intersection / union if union > 0 else 1.0
    precision_denominator = true_positive + false_positive
    recall_denominator = true_positive + false_negative

    precision = true_positive / precision_denominator if precision_denominator > 0 else 1.0
    recall = true_positive / recall_denominator if recall_denominator > 0 else 1.0

    return {
        "pixel_dice": float(dice),
        "pixel_iou": float(iou),
        "pixel_precision": float(precision),
        "pixel_recall": float(recall)
    }

def calculate_instance_metrics(prediction_instances, target_instances, iou_threshold=PQ_MATCH_IOU):
    num_predictions = len(prediction_instances)
    num_targets = len(target_instances)

    if num_predictions == 0 and num_targets == 0:
        return {"pq": 1.0, "sq": 1.0, "rq": 1.0, "tp_instances": 0, "fp_instances": 0, "fn_instances": 0, "one_to_many": 0, "many_to_one": 0, "matched_iou_sum": 0.0}
    if num_predictions == 0:
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "tp_instances": 0, "fp_instances": 0, "fn_instances": num_targets, "one_to_many": 0, "many_to_one": 0, "matched_iou_sum": 0.0}
    if num_targets == 0:
        return {"pq": 0.0, "sq": 0.0, "rq": 0.0, "tp_instances": 0, "fp_instances": num_predictions, "fn_instances": 0, "one_to_many": 0, "many_to_one": 0, "matched_iou_sum": 0.0}

    iou_matrix = np.zeros((num_targets, num_predictions), dtype=np.float32)

    for gt_index, gt_mask in enumerate(target_instances):
        gt_area = int(gt_mask.sum())
        if gt_area == 0:
            continue
        for pred_index, pred_mask in enumerate(prediction_instances):
            pred_area = int(pred_mask.sum())
            if pred_area == 0:
                continue
            intersection = int(np.logical_and(gt_mask, pred_mask).sum())
            if intersection == 0:
                continue
            union = gt_area + pred_area - intersection
            iou_matrix[gt_index, pred_index] = intersection / union

    gt_relation_counts = (iou_matrix > iou_threshold).sum(axis=1)
    pred_relation_counts = (iou_matrix > iou_threshold).sum(axis=0)
    one_to_many = int((gt_relation_counts > 1).sum())
    many_to_one = int((pred_relation_counts > 1).sum())

    candidate_pairs = []
    for gt_index in range(num_targets):
        for pred_index in range(num_predictions):
            iou = float(iou_matrix[gt_index, pred_index])
            if iou > iou_threshold:
                candidate_pairs.append((iou, gt_index, pred_index))

    candidate_pairs.sort(key=lambda item: item[0], reverse=True)
    matched_gt, matched_predictions, matched_ious = set(), set(), []

    for iou, gt_index, pred_index in candidate_pairs:
        if gt_index in matched_gt or pred_index in matched_predictions:
            continue
        matched_gt.add(gt_index)
        matched_predictions.add(pred_index)
        matched_ious.append(iou)

    tp = len(matched_ious)
    fp = num_predictions - tp
    fn = num_targets - tp
    matched_iou_sum = float(np.sum(matched_ious))

    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = matched_iou_sum / denominator if denominator > 0 else 0.0
    sq = matched_iou_sum / tp if tp > 0 else 0.0
    rq = tp / denominator if denominator > 0 else 0.0

    return {
        "pq": float(pq), "sq": float(sq), "rq": float(rq),
        "tp_instances": tp, "fp_instances": fp, "fn_instances": fn,
        "one_to_many": one_to_many, "many_to_one": many_to_one, "matched_iou_sum": matched_iou_sum
    }


In [ ]:
def load_validation_images(rows):
    images, valid_rows = [], []
    for row in rows:
        image_path = TRAIN_IMAGE_DIR / row["file_name"]
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"[WARNING] Could not read: {image_path}", flush=True)
            continue
        images.append(image)
        valid_rows.append(row)
    return valid_rows, images

def _worker_evaluate_validation_sample(task):
    configure_worker_process()
    global _local_val_coco

    if "_local_val_coco" not in globals():
        import sys
        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, "w")
        _local_val_coco = COCO(task["annotation_path"])
        sys.stdout = old_stdout

    image_id = task["image_id"]
    file_name = task["file_name"]
    image_shape = tuple(task["image_shape"])
    polygons = task["polygons"]
    confidences = task["confidences"]

    target_instances = []
    annotation_ids = _local_val_coco.getAnnIds(imgIds=[image_id])
    annotations = _local_val_coco.loadAnns(annotation_ids)

    for annotation in annotations:
        mask = _local_val_coco.annToMask(annotation).astype(bool)
        if mask.any():
            target_instances.append(mask)

    target_union = np.zeros(image_shape, dtype=bool)
    for target_instance in target_instances:
        target_union |= target_instance

    outputs = {}
    for threshold in CONFIDENCE_CANDIDATES:
        prediction_union = np.zeros(image_shape, dtype=np.uint8)
        selected_predictions = 0

        for polygon, confidence in zip(polygons, confidences):
            if confidence < threshold:
                continue
            polygon = np.asarray(polygon, dtype=np.float32)
            if len(polygon) < 3:
                continue
            cv2.fillPoly(prediction_union, [np.round(polygon).astype(np.int32)], 1)
            selected_predictions += 1

        prediction_union = prediction_union.astype(bool)
        prediction_instances = split_into_filaments(prediction_union, MIN_COMPONENT_AREA)
        pixel_metrics = calculate_pixel_metrics(prediction_union, target_union)
        instance_metrics = calculate_instance_metrics(prediction_instances, target_instances)

        outputs[threshold] = {
            "file_name": file_name,
            "pixel_dice": pixel_metrics["pixel_dice"],
            "pixel_iou": pixel_metrics["pixel_iou"],
            "pixel_precision": pixel_metrics["pixel_precision"],
            "pixel_recall": pixel_metrics["pixel_recall"],
            "pq": instance_metrics["pq"],
            "sq": instance_metrics["sq"],
            "rq": instance_metrics["rq"],
            "tp_instances": instance_metrics["tp_instances"],
            "fp_instances": instance_metrics["fp_instances"],
            "fn_instances": instance_metrics["fn_instances"],
            "predicted_instances": len(prediction_instances),
            "ground_truth_instances": len(target_instances),
            "one_to_many": instance_metrics["one_to_many"],
            "many_to_one": instance_metrics["many_to_one"],
            "selected_predictions": selected_predictions
        }

    return outputs

In [ ]:
# Force garbage collection before allocating validation arrays
gc.collect()
require_ram_headroom("validation", minimum_gib=RAM_RESERVE_GIB + 2.0)
validation_start = time.perf_counter()
minimum_confidence = min(CONFIDENCE_CANDIDATES)

VALIDATION_METHODS = ["YOLO only"]
if "refiner" in globals():
    VALIDATION_METHODS.append("YOLO + crop refiner")
validation_examples = []
threshold_results = {
    method: {threshold: {"per_image": []} for threshold in CONFIDENCE_CANDIDATES}
    for method in VALIDATION_METHODS
}
val_rows = val_records[["image_id", "file_name"]].to_dict(orient="records")
total_validation_records = len(val_rows)
refiner_validation_seconds = 0.0

print("\n" + "=" * 90)
print("                 PARALLEL VALIDATION")
print("=" * 90)
print(f"Validation records : {total_validation_records:,}")
print(f"Methods            : {', '.join(VALIDATION_METHODS)}")
print(f"CPU workers        : {NUM_WORKERS}")
print(f"GPU batch size     : {VAL_BATCH_SIZE}")
print("CPU workload       : polygon rasterization + connected components + metrics")
print("=" * 90, flush=True)

pool = multiprocessing.Pool(processes=NUM_WORKERS, initializer=configure_worker_process)
pending = []
def collect_validation_result(method, async_result):
    result = async_result.get()
    for threshold in CONFIDENCE_CANDIDATES:
        threshold_results[method][threshold]["per_image"].append(result[threshold])

try:
    for batch_start in range(0, total_validation_records, VAL_BATCH_SIZE):
        batch_rows = val_rows[batch_start:batch_start + VAL_BATCH_SIZE]
        loaded_rows, images = load_validation_images(batch_rows)
        if not images:
            continue

        # One YOLO pass produces both baselines; the refiner only adds a crop pass.
        raw_batch_instances = predict_instances_batch(
            images, min_confidence=minimum_confidence, apply_refiner=False
        )
        if "refiner" in globals():
            refine_start = time.perf_counter()
            refined_batch_instances = [
                refine_instances_for_image(image, instances)
                for image, instances in zip(images, raw_batch_instances)
            ]
            refiner_validation_seconds += time.perf_counter() - refine_start
        else:
            refined_batch_instances = raw_batch_instances

        for row, image, raw_instances, refined_instances in zip(
            loaded_rows, images, raw_batch_instances, refined_batch_instances
        ):
            for method, instances in (
                ("YOLO only", raw_instances),
                ("YOLO + crop refiner", refined_instances),
            ):
                if method not in threshold_results:
                    continue
                polygons, confidences = [], []
                for instance in instances:
                    contours, _ = cv2.findContours(
                        instance["mask"].astype(np.uint8),
                        cv2.RETR_EXTERNAL,
                        cv2.CHAIN_APPROX_SIMPLE,
                    )
                    if not contours:
                        continue
                    contour = max(contours, key=cv2.contourArea)
                    if len(contour) < 3:
                        continue
                    polygons.append(contour.reshape(-1, 2).astype(np.float32))
                    confidences.append(float(instance["confidence"]))
                task = {
                    "annotation_path": str(ANNOTATION_PATH),
                    "image_id": row["image_id"],
                    "file_name": row["file_name"],
                    "image_shape": image.shape,
                    "polygons": polygons,
                    "confidences": confidences,
                }
                pending.append((method, pool.apply_async(_worker_evaluate_validation_sample, (task,))))
                if len(pending) >= MAX_PENDING_VALIDATION:
                    method_to_collect, async_result = pending.pop(0)
                    collect_validation_result(method_to_collect, async_result)

        if len(validation_examples) < 4:
            for row, image, raw_instances, refined_instances in zip(
                loaded_rows, images, raw_batch_instances, refined_batch_instances
            ):
                if len(validation_examples) >= 4:
                    break
                target = np.zeros(image.shape, dtype=bool)
                for annotation in coco.loadAnns(coco.getAnnIds(imgIds=[row["image_id"]])):
                    target |= coco.annToMask(annotation).astype(bool)
                validation_examples.append({
                    "image": image.copy(),
                    "target": target.copy(),
                    "instances": [
                        {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in inst.items()}
                        for inst in raw_instances
                    ],
                    "refined_instances": [
                        {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in inst.items()}
                        for inst in refined_instances
                    ],
                    "file_name": row["file_name"],
                })

        processed = min(batch_start + len(loaded_rows), total_validation_records)
        print(f"{processed:,}/{total_validation_records:,} records | pending CPU jobs: {len(pending)}", flush=True)
        del loaded_rows, images, raw_batch_instances, refined_batch_instances
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n[CPU] Draining remaining validation workers...", flush=True)
    while pending:
        method, async_result = pending.pop(0)
        collect_validation_result(method, async_result)
finally:
    pool.close()
    pool.join()

validation_time = time.perf_counter() - validation_start
threshold_rows = []
for method in VALIDATION_METHODS:
    for threshold in CONFIDENCE_CANDIDATES:
        per_image_df = pd.DataFrame(threshold_results[method][threshold]["per_image"])
        if per_image_df.empty:
            continue
        threshold_rows.append({
            "method": method, "confidence": threshold,
            "PQ": per_image_df["pq"].mean(), "SQ": per_image_df["sq"].mean(),
            "RQ": per_image_df["rq"].mean(), "Mean Dice": per_image_df["pixel_dice"].mean(),
            "Mean IoU": per_image_df["pixel_iou"].mean(),
            "Mean Precision": per_image_df["pixel_precision"].mean(),
            "Mean Recall": per_image_df["pixel_recall"].mean(),
            "TP": int(per_image_df["tp_instances"].sum()),
            "FP": int(per_image_df["fp_instances"].sum()),
            "FN": int(per_image_df["fn_instances"].sum()),
            "Predicted Instances": int(per_image_df["predicted_instances"].sum()),
            "GT Instances": int(per_image_df["ground_truth_instances"].sum()),
            "One-to-Many": int(per_image_df["one_to_many"].sum()),
            "Many-to-One": int(per_image_df["many_to_one"].sum()),
        })

threshold_summary = pd.DataFrame(threshold_rows).sort_values("PQ", ascending=False).reset_index(drop=True)
if threshold_summary.empty:
    raise RuntimeError("Validation produced no metrics; check the validation split and image paths.")
BEST_METHOD = str(threshold_summary.iloc[0]["method"])
BEST_CONFIDENCE = float(threshold_summary.iloc[0]["confidence"])
BEST_CONFIDENCE_BY_METHOD = {
    str(method): float(group.loc[group["PQ"].idxmax(), "confidence"])
    for method, group in threshold_summary.groupby("method")
}
USE_REFINER = BEST_METHOD == "YOLO + crop refiner"

print("\n" + "=" * 100)
print("                    VALIDATION RESULTS (RANKED BY PQ)")
print("=" * 100)
display(threshold_summary.round(4))
print(f"\nSelected method: {BEST_METHOD}")
print(f"Selected YOLO confidence: {BEST_CONFIDENCE:.2f}")
print(f"Best validation PQ: {threshold_summary.iloc[0]['PQ']:.4f}")
print(f"Validation time, both methods: {validation_time:.1f}s")
print(f"Refiner inference time on validation: {refiner_validation_seconds:.1f}s")
print("=" * 100, flush=True)


## 9. Generate the competition submission

Test inference runs one native-resolution image at a time. CPU connected-component extraction and RLE encoding use at most four single-threaded workers, and rows are written incrementally to the CSV.

In [ ]:
import queue

def _submission_worker_optimized(task, result_queue):
    """
    Worker process target. Processes the mask and pushes rows 
    straight to a thread-safe queue to avoid main-thread stalling.
    """
    combined_mask = task["combined_mask"]
    path_stem = task["path_stem"]
    components = split_into_filaments(combined_mask, MIN_COMPONENT_AREA)

    rows = []
    for filament_counter, component in enumerate(components, start=1):
        rows.append((f"{path_stem}_{filament_counter}", mask_to_rle(component)))
    
    # Push the result back to the main thread
    result_queue.put(rows)

def create_submission_parallel(output_path=SUBMISSION_PATH):
    global SUBMISSION_INFERENCE_SECONDS
    paths = get_test_image_paths()
    if not paths:
        raise FileNotFoundError(f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}")

    require_ram_headroom("submission inference", minimum_gib=RAM_RESERVE_GIB + 2.0)
    total_images = len(paths)
    start_time = time.perf_counter()

    print(f"[START] Test images: {total_images:,}\n[CONFIG] GPU batch: {VAL_BATCH_SIZE}\n[CONFIG] CPU workers: {NUM_WORKERS}", flush=True)
    
    # Use a Manager to create a thread/process-safe Queue
    manager = multiprocessing.Manager()
    result_queue = manager.Queue()
    
    cpu_pool = multiprocessing.Pool(processes=NUM_WORKERS, initializer=configure_worker_process)

    with open(output_path, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["filament_id", "segmentation_rle"])

        total_rows_written = 0
        tasks_submitted = 0
        tasks_resolved = 0
        seen_filament_ids = set()

        try:
            for batch_start in range(0, total_images, VAL_BATCH_SIZE):
                batch_paths = paths[batch_start:batch_start + VAL_BATCH_SIZE]
                images = []

                for path in batch_paths:
                    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
                    if image is None:
                        raise OSError(f"Could not read test image: {path}")
                    images.append(image)

                # Heavy GPU inference step
                batch_instances = predict_instances_batch(images, min_confidence=BEST_CONFIDENCE)

                for path, image, instances in zip(batch_paths, images, batch_instances):
                    combined = union_instances(instances, BEST_CONFIDENCE)
                    if combined is None:
                        combined = np.zeros(image.shape, dtype=bool)

                    # Dispatch to pool completely asynchronously
                    cpu_pool.apply_async(
                        _submission_worker_optimized, 
                        args=({"combined_mask": combined, "path_stem": path.stem}, result_queue)
                    )
                    tasks_submitted += 1

                # Flush out whatever CPU work has finished up to this point WITHOUT blocking
                while True:
                    try:
                        worker_rows = result_queue.get_nowait()
                        for fid, rle in worker_rows:
                            if fid in seen_filament_ids:
                                raise ValueError(f"Duplicate filament ID detected: {fid}")
                            seen_filament_ids.add(fid)
                            writer.writerow([fid, rle])
                            total_rows_written += 1
                        tasks_resolved += 1
                    except queue.Empty:
                        break # No more completed tasks at this exact millisecond

                processed = min(batch_start + len(batch_paths), total_images)
                print(f"Inference Progress: {processed:,}/{total_images:,} | Active CPU Tasks: {tasks_submitted - tasks_resolved}", flush=True)
                
                del images, batch_instances
                gc.collect()

            # Clean up the pool so workers don't accept new tasks
            cpu_pool.close()
            
            # Block and drain the remaining finished items out of the queue
            print("\nFinishing remaining CPU post-processing tasks...", flush=True)
            while tasks_resolved < tasks_submitted:
                worker_rows = result_queue.get() # Blocking get is safe here because pool is closing
                for fid, rle in worker_rows:
                    if fid in seen_filament_ids:
                        raise ValueError(f"Duplicate filament ID detected: {fid}")
                    seen_filament_ids.add(fid)
                    writer.writerow([fid, rle])
                    total_rows_written += 1
                tasks_resolved += 1

            cpu_pool.join()

        finally:
            # Safeguard to ensure pool terminates if an exception throws early
            cpu_pool.close()
            cpu_pool.terminate()
            cpu_pool.join()

    elapsed = time.perf_counter() - start_time
    SUBMISSION_INFERENCE_SECONDS = elapsed
    print(f"\n[SUCCESS] Submission complete!\nProcessed: {total_images:,} images\nFilament rows written: {total_rows_written:,}\nRuntime: {elapsed / 60:.2f} min\nSaved: {output_path}", flush=True)
    return pd.read_csv(output_path, nrows=5) if total_rows_written > 0 else None


## 10. Submission audit and final validation report

In [ ]:
if CREATE_SUBMISSION:
    create_submission_parallel()
    
    audit_columns = None
    audit_row_count = 0
    audit_null_count = 0
    duplicate_ids = 0
    empty_rle_count = 0
    seen_ids = set()
    sample_rle_valid = False

    

    with open(SUBMISSION_PATH, newline="", encoding="utf-8") as audit_file:
        reader = csv.DictReader(audit_file)
        audit_columns = reader.fieldnames
        for row in reader:
            audit_row_count += 1
            if audit_row_count == 1:
                sample_rle_valid = bool(row.get("segmentation_rle"))

            if any(value is None for value in row.values()):
                audit_null_count += 1

            filament_id = row.get("filament_id")
            rle = row.get("segmentation_rle")
            if not filament_id or not rle:
                empty_rle_count += 1
            if filament_id in seen_ids:
                duplicate_ids += 1
            elif filament_id:
                seen_ids.add(filament_id)
else:
    audit_columns = ["filament_id", "segmentation_rle"]
    audit_row_count = 0
    audit_null_count = 0
    duplicate_ids = 0
    empty_rle_count = 0
    sample_rle_valid = False

print("--- Submission File Integrity Audit (streamed) ---")
print(f"Total rows: {audit_row_count:,}")
print(f"Null values: {audit_null_count:,}")
print(f"Duplicate filament IDs: {duplicate_ids:,}")
print(f"Empty IDs/RLE strings: {empty_rle_count:,}")
print(f"Sample RLE check: {sample_rle_valid}")

submission_valid = (
    set(audit_columns or []) == {"filament_id", "segmentation_rle"}
    and audit_null_count == 0
    and duplicate_ids == 0
    and empty_rle_count == 0
    and (sample_rle_valid or audit_row_count == 0)
)

print("\n" + ("[VERIFIED] Submission structure passed the local audit." if submission_valid else "[WARNING] Submission structure needs inspection."))

best_row = threshold_summary[
    (threshold_summary["method"] == BEST_METHOD)
    & (threshold_summary["confidence"] == BEST_CONFIDENCE)
].iloc[0]

final_metrics = pd.DataFrame(
    {
        "Metric": [
            "Panoptic Quality (PQ)", "Segmentation Quality (SQ)",
            "Recognition Quality (RQ)", "Mean Dice", "Mean IoU",
            "Mean Precision", "Mean Recall", "True Positives",
            "False Positives", "False Negatives", "Predicted Instances",
            "Ground-Truth Instances", "One-to-Many (Fragmentation)",
            "Many-to-One (Over-Merging)", "Selected Prediction Method", "Selected Confidence",
        ],
        "Value": [
            best_row["PQ"], best_row["SQ"], best_row["RQ"],
            best_row["Mean Dice"], best_row["Mean IoU"],
            best_row["Mean Precision"], best_row["Mean Recall"],
            int(best_row["TP"]), int(best_row["FP"]), int(best_row["FN"]),
            int(best_row["Predicted Instances"]), int(best_row["GT Instances"]),
            int(best_row["One-to-Many"]), int(best_row["Many-to-One"]),
            BEST_METHOD, BEST_CONFIDENCE,
        ],
    }
)

print("\n" + "=" * 75)
print("                 FINAL VALIDATION REPORT")
print("=" * 75)
display(final_metrics.round(4))

selected_per_image = pd.DataFrame(
    threshold_results[BEST_METHOD][BEST_CONFIDENCE]["per_image"]
)
distribution = (
    selected_per_image[["pixel_dice", "pixel_iou", "pq", "sq", "rq"]]
    .agg(["mean", "median", "std", "min", "max"])
    .T
)
print("\n--- Per-image metric distribution ---")
display(distribution.round(4))

best_mask_map50 = (
    float(history["metrics/mAP50(M)"].max())
    if "metrics/mAP50(M)" in history.columns else np.nan
)
best_mask_map50_95 = (
    float(history["metrics/mAP50-95(M)"].max())
    if "metrics/mAP50-95(M)" in history.columns else np.nan
)

diagnostics = pd.DataFrame(
    {
        "Diagnostic": [
            "YOLO best mask mAP50", "YOLO best mask mAP50-95",
            "Validation records", "Submission rows", "Submission valid",
            "YOLO training time (min)", "Refiner training time (min)",
            "Validation time (min)", "Refiner validation inference (s)",
            "Submission inference time (min)",
        ],
        "Value": [
            best_mask_map50, best_mask_map50_95,
            len(selected_per_image), audit_row_count, submission_valid,
            training_time / 60, refiner_training_seconds / 60,
            validation_time / 60, refiner_validation_seconds,
            globals().get("SUBMISSION_INFERENCE_SECONDS", np.nan) / 60,
        ],
    }
)
print("\n--- Model / submission diagnostics ---")
display(diagnostics)

final_metrics.to_csv(FINAL_METRICS_PATH, index=False)
selected_per_image.to_csv(PER_IMAGE_METRICS_PATH, index=False)
print(f"Saved validation summary: {FINAL_METRICS_PATH}")
print(f"Saved per-image metrics: {PER_IMAGE_METRICS_PATH}")

## 11. Threshold visualization and validation examples

Threshold selection is based on **PQ**.

In [ ]:
fig, axis = plt.subplots(figsize=(7, 4.5))
for method, method_rows in threshold_summary.groupby("method"):
    method_rows = method_rows.sort_values("confidence")
    axis.plot(method_rows["confidence"], method_rows["PQ"], marker="o", label=method)
axis.axvline(BEST_CONFIDENCE, linestyle="--", label=f"Selected confidence: {BEST_CONFIDENCE:.2f}")
axis.set(title="Validation PQ vs YOLO confidence", xlabel="YOLO instance confidence", ylabel="Panoptic Quality (PQ)", ylim=(0, 1))
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
plt.show()

for example in validation_examples:
    image = example["image"]
    target = example["target"]
    raw_instances = example["instances"]
    refined_instances = example["refined_instances"]
    raw_confidence = BEST_CONFIDENCE_BY_METHOD["YOLO only"]
    refined_confidence = BEST_CONFIDENCE_BY_METHOD.get("YOLO + crop refiner", BEST_CONFIDENCE)

    raw_prediction = union_instances(raw_instances, raw_confidence)
    refined_prediction = union_instances(refined_instances, refined_confidence)
    if raw_prediction is None:
        raw_prediction = np.zeros_like(target, dtype=bool)
    if refined_prediction is None:
        refined_prediction = np.zeros_like(target, dtype=bool)
    raw_count = len(build_final_prediction_instances(raw_instances, raw_confidence))
    refined_count = len(build_final_prediction_instances(refined_instances, refined_confidence))

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Input image")
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")
    axes[2].imshow(raw_prediction, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"YOLO only ? {raw_count} components")
    axes[3].imshow(refined_prediction, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title(f"YOLO + refiner ? {refined_count} components")
    for axis in axes:
        axis.axis("off")
    fig.suptitle(example["file_name"])
    fig.tight_layout()
    plt.show()


## 12. Display a labeled prediction on an unseen test image

The test image is used only for inference/visualization. No test ground truth is accessed.

In [ ]:
def plot_labeled_prediction(image, instances, title):
    combined = union_instances(instances, BEST_CONFIDENCE)
    components = split_into_filaments(combined) if combined is not None else []

    fig, axis = plt.subplots(figsize=(9, 9))
    axis.imshow(image, cmap="gray")

    for filament_index, component in enumerate(components, start=1):
        axis.contour(component.astype(np.uint8), levels=[0.5], linewidths=1.2)
        moments = cv2.moments(component.astype(np.uint8))

        if moments["m00"]:
            center_x = moments["m10"] / moments["m00"]
            center_y = moments["m01"] / moments["m00"]
        else:
            ys, xs = np.where(component)
            center_x, center_y = xs.mean(), ys.mean()

        area = int(component.sum())
        axis.text(
            center_x, center_y, f"F{filament_index}\n{area:,} px", fontsize=8, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.75)
        )

    axis.set_title(f"{title} · {len(components)} predicted filament(s)")
    axis.axis("off")
    fig.tight_layout()
    plt.show()

test_paths = get_test_image_paths()
if not test_paths:
    raise FileNotFoundError("No test images were found.")

test_path = test_paths[0]
test_image = cv2.imread(str(test_path), cv2.IMREAD_GRAYSCALE)
if test_image is None:
    raise OSError(f"Could not read test image: {test_path}")

test_instances = predict_instances(test_image, min_confidence=BEST_CONFIDENCE)
print(f"Displaying labeled prediction for test image: {test_path.name}")
plot_labeled_prediction(test_image, test_instances, f"Test prediction · {test_path.name} · confidence {BEST_CONFIDENCE:.2f}")

## 13. Experiment summary

### Main outputs

- `yolo_filament_dataset/` - YOLO-format links/labels and adaptive disk cache
- `yolo_runs/yolo26s_filament_full_resolution/weights/best_pq.pt` - selected YOLO checkpoint by callback PQ
- `yolo_runs/yolo26s_filament_full_resolution/pq_callback_metrics.csv` - periodic PQ callback history
- `yolo_runs/yolo26s_filament_full_resolution/weights/refiner.pth` - trained CropUNet refiner
- `solar_filament_yolo26s_submission.csv` - streamed competition submission
- `solar_filament_yolo26s_validation_metrics.csv` - validation summary
- `solar_filament_yolo26s_per_image_metrics.csv` - per-image metrics

### Resource plan

| Stage | GPU | CPU / memory control |
|---|---|---|
| Training | Both T4 GPUs when available; global batch 2 | 2 loader workers per GPU (4 total), AMP, adaptive disk cache |
| Dataset build | None | 4 single-threaded workers; disk and RAM preflight |
| Validation | GPU 0 | Batch 1; at most 4 metric workers and 8 queued jobs |
| Submission | GPU 0 | Batch 1; 4 RLE workers; CSV written row by row |

The disk preflight estimates a full RGB cache at 2048 x 2048, worst-case linked-image fallbacks, and training outputs. It enables disk caching only when that plan fits the actual working volume; otherwise it disables caching and preserves at least 4 GiB free. RAM checks stop expensive stages when available headroom is too low.